# TUGAS 4.1: PREPROCESSING TEKS BAHASA INDONESIA
**Sistem Temu Kembali Informasi**

* **Nama    :** Washiatul Akmal
* **NIM     :** 240210501050
* **Kelas   :** Sistem Temu Kembali | Tekom Pilihan B 24

In [9]:
import re
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# 1. Dataset 5 Dokumen Pengumuman & Berita Kampus (Tema: Komputer / Teknologi / Pendidikan)
dokumen_korpus = [
    {
        "id": "DOK_01",
        "judul": "Pelatihan AI Laboratorium Komputer",
        "teks": "Fakultas Teknik UNM menyelenggarakan pelatihan kecerdasan buatan dan machine learning untuk mahasiswa di laboratorium komputer pada tanggal 12 Oktober 2026!"
    },
    {
        "id": "DOK_02",
        "judul": "Pembaruan Infrastruktur Jaringan",
        "teks": "Tim teknisi jurusan teknik informatika sedang meningkatkan kecepatan akses internet dan keamanan sistem jaringan nirkabel di area gedung perkuliahan."
    },
    {
        "id": "DOK_03",
        "judul": "Sosialisasi Portal e-Skripsi",
        "teks": "Pengelola sistem informasi mengumumkan bahwa seluruh mahasiswa tingkat akhir wajib mengunggah naskah skripsi melalui aplikasi web resmi kampus."
    },
    {
        "id": "DOK_04",
        "judul": "Pameran Inovasi Internet of Things (IoT)",
        "teks": "Mahasiswa teknik komputer memamerkan alat pengering otomatis berbasis mikrokontroler ESP32 serta sensor pintar dalam kegiatan pameran teknologi tahunan."
    },
    {
        "id": "DOK_05",
        "judul": "Workshop Pemrograman Citra Digital",
        "teks": "Laboratorium rekayasa perangkat lunak mengadakan pelatihan pemrograman bahasa Python untuk segmentasi citra medis dan klasifikasi data digital secara daring."
    }
]

print("=== DAFTAR 5 DOKUMEN ASLI ===")
for d in dokumen_korpus:
    print(f"[{d['id']}] {d['judul']}")
    print(f"Teks: {d['teks']}\n")

=== DAFTAR 5 DOKUMEN ASLI ===
[DOK_01] Pelatihan AI Laboratorium Komputer
Teks: Fakultas Teknik UNM menyelenggarakan pelatihan kecerdasan buatan dan machine learning untuk mahasiswa di laboratorium komputer pada tanggal 12 Oktober 2026!

[DOK_02] Pembaruan Infrastruktur Jaringan
Teks: Tim teknisi jurusan teknik informatika sedang meningkatkan kecepatan akses internet dan keamanan sistem jaringan nirkabel di area gedung perkuliahan.

[DOK_03] Sosialisasi Portal e-Skripsi
Teks: Pengelola sistem informasi mengumumkan bahwa seluruh mahasiswa tingkat akhir wajib mengunggah naskah skripsi melalui aplikasi web resmi kampus.

[DOK_04] Pameran Inovasi Internet of Things (IoT)
Teks: Mahasiswa teknik komputer memamerkan alat pengering otomatis berbasis mikrokontroler ESP32 serta sensor pintar dalam kegiatan pameran teknologi tahunan.

[DOK_05] Workshop Pemrograman Citra Digital
Teks: Laboratorium rekayasa perangkat lunak mengadakan pelatihan pemrograman bahasa Python untuk segmentasi citra medis 

In [10]:
# Inisialisasi engine Sastrawi
stop_factory = StopWordRemoverFactory()
stopword_remover = stop_factory.create_stop_word_remover()

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def preprocess_text(text):
    """
    Pipeline pemrosesan teks berurutan:
    1. Case folding
    2. Cleaning (hapus angka & tanda baca)
    3. Tokenisasi
    4. Stopwords removal (Sastrawi)
    5. Stemming (Sastrawi)
    """
    # 1. Case Folding
    text_lower = text.lower()
    
    # 2. Cleaning
    text_clean = re.sub(r'[^a-z\s]', ' ', text_lower)
    
    # 3. Tokenisasi
    tokens_raw = text_clean.split()
    
    # 4. Stopwords Removal
    text_filtered = stopword_remover.remove(" ".join(tokens_raw))
    tokens_filtered = text_filtered.split()
    
    # 5. Stemming
    tokens_stemmed = [stemmer.stem(t) for t in tokens_filtered if t.strip() != '']
    
    return tokens_stemmed

In [11]:
# Eksekusi fungsi pada ke-5 dokumen
hasil_evaluasi = []

for item in dokumen_korpus:
    teks_mentah = item["teks"]
    token_awal = teks_mentah.split()
    token_hasil = preprocess_text(teks_mentah)
    
    hasil_evaluasi.append({
        "Dokumen": item["id"],
        "Sebelum Preprocessing": teks_mentah,
        "Sesudah Preprocessing": " ".join(token_hasil),
        "Token Sebelum": len(token_awal),
        "Token Setelah": len(token_hasil)
    })

# Tampilkan perbandingan sebelum dan sesudah secara lengkap
df_perbandingan = pd.DataFrame(hasil_evaluasi)[["Dokumen", "Sebelum Preprocessing", "Sesudah Preprocessing"]]
print("=== PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING ===")
for _, row in df_perbandingan.iterrows():
    print(f"ID Dokumen : {row['Dokumen']}")
    print(f"Sebelum    : {row['Sebelum Preprocessing']}")
    print(f"Sesudah    : {row['Sesudah Preprocessing']}")
    print("-" * 80)

=== PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING ===
ID Dokumen : DOK_01
Sebelum    : Fakultas Teknik UNM menyelenggarakan pelatihan kecerdasan buatan dan machine learning untuk mahasiswa di laboratorium komputer pada tanggal 12 Oktober 2026!
Sesudah    : fakultas teknik unm selenggara latih cerdas buat machine learning mahasiswa laboratorium komputer tanggal oktober
--------------------------------------------------------------------------------
ID Dokumen : DOK_02
Sebelum    : Tim teknisi jurusan teknik informatika sedang meningkatkan kecepatan akses internet dan keamanan sistem jaringan nirkabel di area gedung perkuliahan.
Sesudah    : tim teknisi jurus teknik informatika sedang tingkat cepat akses internet aman sistem jaring nirkabel area gedung kuliah
--------------------------------------------------------------------------------
ID Dokumen : DOK_03
Sebelum    : Pengelola sistem informasi mengumumkan bahwa seluruh mahasiswa tingkat akhir wajib mengunggah naskah skripsi melalui 

In [12]:
# Hitung reduksi token: ((Token Sebelum - Token Setelah) / Token Sebelum) * 100%
data_reduksi = []
for h in hasil_evaluasi:
    t_awal = h["Token Sebelum"]
    t_akhir = h["Token Setelah"]
    persentase = ((t_awal - t_akhir) / t_awal) * 100
    
    data_reduksi.append({
        "Dokumen": h["Dokumen"],
        "Token Sebelum": t_awal,
        "Token Setelah": t_akhir,
        "Selisih Token": t_awal - t_akhir,
        "Persentase Pengurangan": f"{persentase:.2f}%"
    })

df_reduksi = pd.DataFrame(data_reduksi)
print("=== TABEL EVALUASI JUMLAH TOKEN ===")
display(df_reduksi)

=== TABEL EVALUASI JUMLAH TOKEN ===


,Dokumen,Token Sebelum,Token Setelah,Selisih Token,Persentase Pengurangan
0,DOK_01,20,14,6,30.00%
1,DOK_02,19,17,2,10.53%
2,DOK_03,18,17,1,5.56%
3,DOK_04,18,16,2,11.11%
4,DOK_05,19,16,3,15.79%


### Analisis Dampak Preprocessing terhadap Sistem IR
Tahapan preprocessing memberikan dampak signifikan terhadap peningkatan kualitas data sistem temu kembali informasi (IR) melalui pembersihan derau (*noise*) dan penyeragaman bentuk kata. Proses *cleaning* berhasil mengeliminasi tanda baca serta angka yang tidak memuat nilai relevansi, sementara *stopword removal* secara efektif membuang kata-kata umum berfrekuensi tinggi yang minim daya diskriminatif antar-dokumen. Selain itu, teknik *stemming* berhasil menyatukan bentuk fleksi kata (seperti *meningkatkan* menjadi *tingkat*, dan *pelatihan* menjadi *latih*) ke bentuk dasar yang seragam. Rangkaian proses ini mereduksi dimensi indeks korpus secara substansial, mencegah ledakan kosakata (*vocabulary size*), serta mempercepat pencocokan kueri pencarian dokumen karena sistem tidak lagi terhalang oleh perbedaan imbuhan gramatikal.